<div style="
background-color:#EAEAEA;
padding:20px;
border-left:5px solid #6C757D;
border-radius:6px;">

<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="border:none; vertical-align:top;">

<h1 style="font-size:32px; margin-top:0;">
Master's Thesis
</h1>

<hr style="margin:16px 0 22px 0;">

<p style="font-size:22px; line-height:1.5; margin:0;">
<strong>Master's Degree in Advanced Physics</strong> - 
<strong>Universitat de Val?ncia</strong>
</p>

<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">
This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:
</p>

<div style="
font-size:25px;
font-weight:700;
line-height:1.3;
margin-top:14px;
margin-bottom:26px;">
Fast Simulation of Neutrino Oscillations in Matter
</div>

<p style="font-size:14px; line-height:1.55;">
<strong>Author</strong><br>
Juan Ramon Diaz Santos - 
<a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a>
</p>

<p style="font-size:14px; line-height:1.55;">
<strong>Supervisors</strong><br>
Roberto Ruiz de Austri Bazan ?
<a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>
Michele Lucente ?
<a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a>
</p>

<p style="font-size:14px; line-height:1.55; margin-bottom:0;">
<strong>Date</strong><br>
September 2026
</p>

</td>

<td style="
border:none;
width:230px;
padding-left:25px;
text-align:right;
vertical-align:top;">

<img src="../../logo_uv.png"
     alt="Universitat de Val?ncia"
     style="width:200px; margin-top:5px;">

</td>

</tr>
</table>

</div>

# Validation NuSQuIDS 6: BSM (3+1 Sterile) Solar Neutrino to Earth Detection
---

This notebook is the 3+1-sterile counterpart of `nusquids5_SolarNeutrino_EarthDetection.ipynb` §5: it
cross-checks tpeanuts's sterile-extended Earth regeneration against an independent external
implementation, nuSQuIDS, built from scratch in C++ with no shared code path with tpeanuts.

**Scope note (NSI):** an equivalent NSI-vs-nuSQuIDS comparison was investigated for this notebook but is
not included here. The installed nuSQuIDS 1.13.3 Python bindings expose no NSI-capable class (only the
plain `nuSQUIDS`/`nuSQUIDSAtm`); NSI support in nuSQuIDS exists only as an un-packaged, mu-tau-only C++
example (`examples/NSI/NSI.h`, hardcoded to `numneu=3`, no Python bindings) that is not part of any
distributed build. Building general-epsilon NSI support against this installation would mean writing new
C++ Hamiltonian and pybind11 code, not installing an existing package — out of scope here. External NSI
validation is instead covered analytically, without nuSQuIDS, by extending
`notebooks/validation/physics/BSM/NSI2_lma_dark.ipynb`'s LMA-Dark degeneracy check to the exact
`nsi_lma_dark_esteban2018` preset used in `validation_intrinsic2_BSM_perturvative_vs_numerical.ipynb`.

nuSQuIDS's 3+1 sterile support, by contrast, is native and well-tested (`numneu>=4` in the `nuSQUIDS`
constructor), so this notebook covers Sterile only, split into two parts:

- **Part A — Null-mixing sanity check:** the 3+1 machinery at `theta14=theta24=theta34=0` must reduce
  exactly to the plain Standard Model in both codes, confirming the geometry/unit-convention bridge
  between tpeanuts and nuSQuIDS is correct before trusting it for a genuinely mixed sterile scenario.
- **Part B — Sterile preset comparison:** the same three literature sterile presets used in
  `validation_intrinsic2_BSM_perturvative_vs_numerical.ipynb` §6 (`sterile_3p1_bestfit_giunti2017`,
  `sterile_3p1_benchmark_icecube`, `sterile_3p1_two_flavour_limit`), each compared against nuSQuIDS at a
  handful of representative (E, eta) points.

**Environment note:** unlike every other notebook in this repository, this one requires the nuSQuIDS
Python bindings, importable in-process alongside tpeanuts. On this project's Windows development machine
nuSQuIDS is only installed inside a WSL/Debian virtual environment (`~/hep/opt/corsika8-venv`, Python
3.13) that also contains a live symlink to this repository as its `tpeanuts` package. Run this notebook
with that environment's Jupyter kernel; `NSQ_AVAILABLE` below degrades gracefully (NaNs, no crash) if
nuSQuIDS is not importable in whatever kernel is active.

## Table of Contents

| # | Section |
|---|---------|
| [0](#0.-Theory-Background) | **Theory Background**: cross-check methodology, geometry/convention bridge |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Part-A----Null-Mixing-Sanity-Check) | **Part A — Null-Mixing Sanity Check** |
| [4](#4.-Part-B----Sterile-Preset-Comparison) | **Part B — Sterile Preset Comparison** |
| [5](#5.-Summary) | **Summary** |

## 0. Theory Background

### 0.1 Why an external, independent code matters

`validation_intrinsic2_BSM_perturvative_vs_numerical.ipynb` established that tpeanuts's own perturbative
(analytical) and matrix-exponential (numerical) propagation branches agree with each other for the 3+1
sterile extension. That is strong evidence against *implementation* bugs in either branch, but both
branches build their Hamiltonian through the same shared functions
(`hamiltonian_matter_sterile`, `kinetic_hamiltonian_from_mass_vector`) — a *physics* error common to both
would still pass that check. nuSQuIDS shares no code with tpeanuts at all: its 3+1 sterile Hamiltonian is
implemented independently in C++, so agreement here is evidence the physics itself, not just tpeanuts's
internal consistency, is correct.

### 0.2 Isolating the Earth-propagation step

tpeanuts has no sterile-aware solar MSW production model (the sterile state is never produced by weak
interactions in the solar core — see `validation_intrinsic2...ipynb` §0.4). Reproducing tpeanuts's solar
production physics inside nuSQuIDS as well would introduce a second, unrelated source of disagreement
(solar density-profile and source-distribution differences) on top of what this notebook actually wants
to test: whether the two codes' **3+1 sterile Earth-crossing Hamiltonians** agree. This notebook therefore
computes the incoherent mass-basis weights **once**, with tpeanuts's own (already-validated) active-sector
solar model, zero-pads the sterile entry, and feeds that **identical** starting state into both codes'
Earth propagation.

### 0.3 nuSQuIDS Earth body and the mass-basis chaining trick

As documented in `nusquids5...ipynb` §5, the installed nuSQuIDS 1.13.3 plain `Earth` body exposes no
`MakeTrack`/`MakeTrackWithCosine` convenience constructor; `tpeanuts.external.nusquids.core`'s
`_earth_body_and_track` works around this by computing the physical chord baseline directly,
$L=\max(0,\,-2R_\oplus\cos\theta_z)$, and building an `Earth.Track(L)` from it. nuSQuIDS has no single call
chaining an incoherent mass state through this body either, so `transition_matrix_earth_mass_to_flavour`
propagates each mass eigenstate **separately** (`Set_initial_state(..., Basis.mass)`, one pure eigenstate
at a time) and recombines incoherently — now generalized in this notebook's session to `numneu=4`, with
the fourth (sterile) mass eigenstate propagated exactly like the other three.

### 0.4 Geometry and sign-convention bridge

- **Angle convention:** tpeanuts's nadir angle $\eta$ and nuSQuIDS's $\cos\theta_z$ (zenith cosine) are
  complementary, $\theta_z=\pi-\eta$, so $\cos\theta_z=-\cos\eta$ (same relation used in `nusquids5...ipynb`).
- **Depth simplification:** `probability_earth_massbasis` assumes a surface detector (depth 0); the
  tpeanuts calls in this notebook therefore also use depth 0, unlike the 1000 m used in
  `validation_intrinsic2...ipynb`. The difference is $<0.02\%$ of $R_\oplus$, irrelevant here.
- **Density profile:** tpeanuts uses its own PREM-derived even-power polynomial fit
  (`config.earth_density_file`); nuSQuIDS uses its own bundled PREM table. These are independent
  tabulations of the same physical model, not bit-identical — a real, expected, small residual source
  of disagreement, exactly as already characterized in `nusquids2_solar.ipynb`/`nusquids5...ipynb`.

**Expected results:** Part A agrees with the Standard Model to numerical noise ($\sim10^{-3}$ or better,
limited by nuSQuIDS's own ODE tolerances, not by the sterile extension). Part B agrees at the same
percent-level seen throughout the `nusquids2-5` SM comparisons, with no systematic degradation attributable
specifically to the sterile extension — confirming the 3+1 Hamiltonian construction, not just the
propagation algorithm, is physically correct.

### References

- Argüelles, C. A., Salvado, J. & Weaver, C. N. (2022). *nuSQuIDS: A toolbox for neutrino propagation*.
  Comput. Phys. Commun. 277, 108346.
- C. Giunti, M. Marrone and A. Palazzo, *Combined 3+1 global analysis*, arXiv:1612.01087 (2017).

## 1. Libraries

In [ ]:
from __future__ import annotations

%matplotlib inline
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import (
    to_numpy, save_and_show, compare_probability_grids, nusquids_precision_summary,
)
from tpeanuts.external.nusquids.core import (
    NuSQuIDSConfig, is_available as nusquids_is_available, probability_earth_massbasis,
)
from tpeanuts.util.context import RuntimeContext
from tpeanuts.core.common.oscillation import OscillationParameters
from tpeanuts.config.propagation import PropagationConfig
from tpeanuts.core.common.pmns import PMNSParams
from tpeanuts.core.SM.sm_pmns import PMNS_SM
from tpeanuts.core.SM.sm_mass_spectrum import MassSpectrum_SM
from tpeanuts.medium.solar.profile import SolarProfile
from tpeanuts.medium.solar.probability import solar_probability_mass
from tpeanuts.medium.earth.profile import EarthParameters, EarthProfile
from tpeanuts.medium.earth.probability import earth_probability_state_analytical

## 2. Paths and Configuration

**Expected results:** the package/output directories exist, and `nusquids_is_available()` is True when
run under the WSL nuSQuIDS+tpeanuts kernel described in §0.

In [ ]:
config = load_notebook_config()
ctx = RuntimeContext.resolve(config.device, config.dtype)
OUTPUT_DIR = config.output_dir("validation", "nusquids")
SHOW = config.show_plots
DEPTH_M = 0.0  # matches probability_earth_massbasis's surface-detector assumption (see Section 0.4)

NSQ_AVAILABLE = nusquids_is_available()

sm_oscillation = PropagationConfig.oscillation_parameters_from_preset("_SM_NUFIT52_NO", antinu=False, context=ctx)
sterile_null_oscillation = PropagationConfig.oscillation_parameters_from_preset("sterile_3p1_null_mixing", antinu=False, context=ctx)
STERILE_PRESETS = [
    "sterile_3p1_bestfit_giunti2017",
    "sterile_3p1_benchmark_icecube",
    "sterile_3p1_two_flavour_limit",
]

solar = SolarProfile.default(context=ctx)
earth = EarthProfile(
    params=EarthParameters(profile_perturbative_kwargs={"density_file": str(config.earth_density_file), "tabulated_density": False}),
    context=ctx,
)

E_CHECK_MEV = [2.0, 6.0, 10.0, 14.0]
ETA_CHECK = [0.30, 0.90]

print(f"Device : {ctx.device}   dtype : {ctx.dtype}")
print(f"Output : {OUTPUT_DIR}")
print(f"nuSQuIDS available: {NSQ_AVAILABLE}")


def sm_sector_oscillation(sterile_oscillation):
    """Plain 3-flavour OscillationParameters sharing a sterile preset's SM-sector angles (see Section 0.2)."""
    sm_pmns = PMNS_SM(sterile_oscillation.pmns.params)
    mass_spectrum = MassSpectrum_SM(
        DeltamSq21=sterile_oscillation.mass_spectrum.DeltamSq21,
        DeltamSq3l=sterile_oscillation.mass_spectrum.DeltamSq3l,
    )
    return OscillationParameters(
        pmns=sm_pmns, mass_spectrum=mass_spectrum, antinu=sterile_oscillation.antinu,
    )


def sterile_solar_weights(sterile_oscillation, E_MeV):
    """Active-sector solar production weights, zero-padded for the sterile flavour."""
    sm_osc = sm_sector_oscillation(sterile_oscillation)
    E_t = torch.tensor(E_MeV, dtype=ctx.dtype, device=ctx.device)
    w3 = solar_probability_mass(sm_osc, E_t, solar, "8B")
    return torch.cat([w3, torch.zeros_like(w3[..., :1])], dim=-1)


def nsq_config_from_oscillation(oscillation, *, numneu=3):
    """Build a NuSQuIDSConfig sharing an OscillationParameters' angles/splittings (see nusquids5 §5)."""
    params = oscillation.pmns.params if numneu == 3 else oscillation.pmns.params
    kwargs = dict(
        theta12=float(params.theta12), theta13=float(params.theta13), theta23=float(params.theta23),
        delta_cp=float(params.delta), DeltamSq21=float(oscillation.mass_spectrum.DeltamSq21), DeltamSq3l=float(oscillation.mass_spectrum.DeltamSq3l),
        numneu=numneu,
    )
    if numneu >= 4:
        sp = oscillation.pmns.sterile_params
        kwargs.update(
            theta14=float(sp.theta14), theta24=float(sp.theta24), theta34=float(sp.theta34),
            delta14=float(sp.delta14), delta24=float(sp.delta24), DeltamSq41=float(oscillation.mass_spectrum.DeltamSq41),
        )
    return NuSQuIDSConfig(**kwargs)


def compare_solar_to_earth(oscillation, *, numneu, E_list=E_CHECK_MEV, eta_list=ETA_CHECK, label=""):
    """Run the tpeanuts-vs-nuSQuIDS Earth mass-basis comparison over an (E, eta) point list."""
    nsq_cfg = nsq_config_from_oscillation(oscillation, numneu=numneu)
    prob_cols = [f"P_{i}" for i in range(numneu)]
    tp_rows, nsq_rows = [], []
    for eta_val in eta_list:
        weights = sterile_solar_weights(oscillation, E_list) if numneu == 4 else solar_probability_mass(
            oscillation, torch.tensor(E_list, dtype=ctx.dtype, device=ctx.device), solar, "8B",
        )
        eta_t = torch.tensor(eta_val, dtype=ctx.dtype, device=ctx.device)
        for i_E, E_val in enumerate(E_list):
            E_t = torch.tensor(E_val, dtype=ctx.dtype, device=ctx.device)
            P_tp = to_numpy(earth_probability_state_analytical(weights[i_E], earth, oscillation, E_t, eta_t, DEPTH_M, massbasis=True))
            row_tp = {"eta_rad": eta_val, "E_MeV": E_val}
            row_tp.update({c: float(P_tp[i]) for i, c in enumerate(prob_cols)})
            tp_rows.append(row_tp)

            if NSQ_AVAILABLE:
                cos_zenith = -math.cos(eta_val)
                try:
                    P_nsq = probability_earth_massbasis(
                        E_GeV=E_val * 1.0e-3, cos_zenith=cos_zenith,
                        mass_weights=to_numpy(weights[i_E]), config=nsq_cfg,
                    )
                except Exception as exc:  # pragma: no cover - environment dependent
                    print(f"  [skipped] {label} E={E_val} eta={eta_val}: {exc}")
                    P_nsq = np.full(numneu, float("nan"))
            else:
                P_nsq = np.full(numneu, float("nan"))
            row_nsq = {"eta_rad": eta_val, "E_MeV": E_val}
            row_nsq.update({c: float(P_nsq[i]) for i, c in enumerate(prob_cols)})
            nsq_rows.append(row_nsq)

    tp_df = pd.DataFrame(tp_rows)
    nsq_df = pd.DataFrame(nsq_rows)
    comparison = compare_probability_grids(tp_df, nsq_df, ["eta_rad", "E_MeV"], prob_cols=prob_cols)
    summary = nusquids_precision_summary(comparison)
    print(f"--- {label} ---")
    print(summary.to_string())
    return comparison, summary

## 3. Part A — Null-Mixing Sanity Check

Runs the comparison helper on `sterile_3p1_null_mixing` (all sterile angles at zero, see
`validation_intrinsic2...ipynb` §3) through the full 4-flavour code path in both tpeanuts and nuSQuIDS.
The fourth (sterile) probability column must stay at zero in both, and the first three must match the
plain Standard Model to numerical noise, confirming the bridge (angle convention, geometry, units) is
correct before trusting it for a genuinely mixed sterile scenario in Part B.

**Expected results:** `max_abs_err` at the $10^{-3}$ level or better (nuSQuIDS's own ODE tolerance floor,
see §0.4), `P_3` (sterile) column at floating-point zero in both codes.

In [ ]:
null_comparison, null_summary = compare_solar_to_earth(
    sterile_null_oscillation, numneu=4, label="Null-mixing (sterile_3p1_null_mixing)",
)
if not null_comparison.empty:
    print()
    print("Sterile-flavour probability (P_3), tpeanuts:", null_comparison["P_3_tp"].abs().max())
    if "P_3_nsq" in null_comparison.columns:
        print("Sterile-flavour probability (P_3), nuSQuIDS:", null_comparison["P_3_nsq"].abs().max())

## 4. Part B — Sterile Preset Comparison

Runs the same comparison for the three literature sterile presets used in
`validation_intrinsic2...ipynb` §6, at the same (E, eta) points as Part A.

**Runtime note:** unlike Part A (null mixing, no sterile oscillation), these presets have a genuinely
non-zero `DeltamSq41` (0.3-1.7 eV$^2$) combined with `NuSQuIDSConfig`'s tight default ODE tolerances
(`rel_error=1e-11`, `abs_error=1e-13`, inherited unchanged from `nusquids1-5`). A large `DeltamSq41`
means a short sterile oscillation length, especially at the low end of `E_CHECK_MEV`; nuSQuIDS's adaptive
integrator then needs many small steps to hold those tolerances over the Earth-crossing baseline, so this
cell can take several minutes per preset (confirmed by direct timing during development: Part A's 8
points completed in well under a minute, while a single preset's 8 points in Part B took on the order of
5-10 minutes). This is the same "resolution versus fast oscillation" trade-off already characterized for
tpeanuts's own `nsteps` earlier in this project's session history, now manifesting as nuSQuIDS integrator
slowness rather than tpeanuts aliasing error -- it is not a bug, but it does mean this cell is
**intentionally not fast**; reduce `E_CHECK_MEV`/`ETA_CHECK` or loosen `rel_error`/`abs_error` in
`nsq_config_from_oscillation` first if a quicker run is needed.

**Expected results:** agreement at the same order as Part A's Standard Model baseline; the sterile-flavour
probability (`P_3`) should now be small but non-zero in both codes, reflecting genuine active-sterile
conversion through Earth matter.

In [ ]:
preset_results = {}
for preset_name in STERILE_PRESETS:
    oscillation = PropagationConfig.oscillation_parameters_from_preset(preset_name, context=ctx, antinu=False)
    comparison, summary = compare_solar_to_earth(oscillation, numneu=4, label=preset_name)
    preset_results[preset_name] = (comparison, summary)
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
labels = ["Null-mixing"] + STERILE_PRESETS
max_rel = [null_summary.loc["max_rel_err", "value"]] + [preset_results[p][1].loc["max_rel_err", "value"] for p in STERILE_PRESETS]
max_abs = [null_summary.loc["max_abs_err", "value"]] + [preset_results[p][1].loc["max_abs_err", "value"] for p in STERILE_PRESETS]
axes[0].bar(range(len(labels)), max_abs, color="C0"); axes[0].set_title("Maximum absolute error"); axes[0].set_yscale("log")
axes[1].bar(range(len(labels)), max_rel, color="C1"); axes[1].set_title("Maximum relative error"); axes[1].set_yscale("log")
for ax in axes:
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
    ax.grid(alpha=0.25, axis="y")
fig.suptitle("tpeanuts vs nuSQuIDS: solar-to-Earth-detector, 3+1 sterile")
fig.tight_layout()
save_and_show("vn6_fig1_sterile_solar_bar.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW)

## 5. Summary

In [ ]:
all_rows = [{"case": "Null-mixing (sterile_3p1_null_mixing)", **null_summary["value"].to_dict()}]
for preset_name in STERILE_PRESETS:
    all_rows.append({"case": preset_name, **preset_results[preset_name][1]["value"].to_dict()})
summary_table = pd.DataFrame(all_rows).set_index("case")
display(summary_table)
summary_table.to_csv(OUTPUT_DIR / "vn6_summary.csv")

null_comparison.to_csv(OUTPUT_DIR / "vn6_null_mixing_comparison.csv", index=False)
for preset_name in STERILE_PRESETS:
    preset_results[preset_name][0].to_csv(OUTPUT_DIR / f"vn6_{preset_name}_comparison.csv", index=False)

print("Saved:", OUTPUT_DIR / "vn6_summary.csv")
if NSQ_AVAILABLE:
    print(f"All {len(summary_table)} tpeanuts-vs-nuSQuIDS sterile solar-to-Earth comparisons completed.")
else:
    print("nuSQuIDS was not available in this kernel: comparisons are NaN placeholders (see Section 0 environment note).")